# What is the most efficient unbiased estimator of variance?

In [1]:
# N = 11
# NN = Zmod(N)
# R.<n> = FunctionField(QQ)
# F = CombinatorialFreeModule(R, NN, prefix = 'X')
# X = F.basis()

We define some known sample moments.

In [146]:
μ = var('μ')
def sample_moment1(size):
    return μ

σ = var('σ')
def sample_moment2(size):
    return μ^2 + σ^2/size

γ = var('γ')
def sample_moment3(size):
    return μ^3 + 3*μ*σ^2/size + γ*σ^3/size^2

κ = var('κ')
def sample_moment4(size):
    return μ^4 + 6*μ^2*σ^2/size + 4*γ*μ*σ^3/size^2 + (κ + 3*size)*σ^4/size^3

def sample_moment(m, size):
    if m == 0:
        return 1
    elif m == 1:
        return sample_moment1(size)
    elif m == 2:
        return sample_moment2(size)
    elif m == 3:
        return sample_moment3(size)
    elif m == 4:
        return sample_moment4(size)
    else:
        return var('mu_prime_' + str(m) + '_Xbar')

In [147]:
def get_variables(num, prefix='n_', ring=QQ):
    var_list = ','.join([(prefix + str(n)) for n in range(1, num + 1)])
    gens = ring[var_list].gens()
    return_vars = dict()
    for n in range(0, num):
        return_vars[n + 1] = gens[n]
    return return_vars

In [148]:
def get_regular_variables(num, prefix='n_'):
    var_list = ','.join([(prefix + str(n)) for n in range(1, num + 1)])
    gens = var(var_list)
    return_vars = dict()
    for n in range(0, num):
        return_vars[n + 1] = gens[n]
    return return_vars
z = get_regular_variables(2, prefix='n_')
ring = (z[1]/z[2]).parent()

In [149]:
def get_Xs(num):
    return get_variables(num, prefix='X_', ring=ring)

In [150]:
def get_Xbar(X):
    return sum(X.values()) / len(X)

In [151]:
def expectation(expression):
    "expression should be an element of X[1].parent()"
    result = 0
    for coefficient, monomial in expression:
        new_term = coefficient
        for exponent in monomial.exponents()[0]:
            new_term *= sample_moment(exponent, 1)
        result += new_term
    return result
Xbar10 = get_Xbar(get_Xs(10))
show(expectation(Xbar10))
show(expectation(Xbar10^2))
show(expectation(Xbar10^3).full_simplify())
show(expectation(Xbar10^4).full_simplify())

μ

μ^2 + 1/10*σ^2

1/100*γ*σ^3 + μ^3 + 3/10*μ*σ^2

1/25*γ*μ*σ^3 + 1/1000*(κ + 30)*σ^4 + μ^4 + 3/5*μ^2*σ^2

Validating formulas for sample moments

In [152]:
zeros = [
    (size, [
        (moment, (expectation(Xbar^moment) - sample_moment(moment, size)).full_simplify() * (size / σ)^moment)
        for moment in range(1, 4 + 1)
    ])
    for size in range(2, 10 + 1)
    for Xbar in [get_Xbar(get_Xs(size))]
]
show(zeros)

[(2, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (3, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (4, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (5, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (6, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (7, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (8, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (9, [(1, 0), (2, 0), (3, 0), (4, 0)]),
 (10, [(1, 0), (2, 0), (3, 0), (4, 0)])]

In [153]:
def cross_term_kappa(N):
    X = get_Xs(N)
    Xbar = get_Xbar(X)
    return expectation((X[1]-Xbar)^2*(X[2]-Xbar)^2).factor() / σ^4 * N^3

cross_term_kappa(10)

17*κ + 830

In [154]:
[(m, cross_term_kappa(m)) for m in range(2, 11+1)]

[(2, κ + 6),
 (3, 3*κ + 18),
 (4, 5*κ + 44),
 (5, 7*κ + 90),
 (6, 9*κ + 162),
 (7, 11*κ + 266),
 (8, 13*κ + 408),
 (9, 15*κ + 594),
 (10, 17*κ + 830),
 (11, 19*κ + 1122)]

In [155]:
# Build a hypothesis difference
[(m, (cross_term_kappa(m) - (2*m-3)*κ)/m) for m in range(2, 11+1)]

[(2, 3),
 (3, 6),
 (4, 11),
 (5, 18),
 (6, 27),
 (7, 38),
 (8, 51),
 (9, 66),
 (10, 83),
 (11, 102)]

In [156]:
# Build a hypothesis difference
[(m, (cross_term_kappa(m) - (2*m-3)*κ)/m - m^2 + 2*m - 3) for m in range(2, 11+1)]

[(2, 0),
 (3, 0),
 (4, 0),
 (5, 0),
 (6, 0),
 (7, 0),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 0)]

In [157]:
# Hypothesis
[(m, (2*m-3)*κ + m*(m^2 - 2*m + 3)) for m in range(2, 11+1)]

[(2, κ + 6),
 (3, 3*κ + 18),
 (4, 5*κ + 44),
 (5, 7*κ + 90),
 (6, 9*κ + 162),
 (7, 11*κ + 266),
 (8, 13*κ + 408),
 (9, 15*κ + 594),
 (10, 17*κ + 830),
 (11, 19*κ + 1122)]

In [158]:
e3 = expectation((X[1]-Xbar)^4)
e3.factor()

10/1331*(91*κ + 330)*σ^4

In [159]:
def diagonal_term_kappa(N):
    X = get_Xs(N)
    Xbar = get_Xbar(X)
    return expectation((X[1]-Xbar)^4).factor() / σ^4 * N^3 / (N-1)

diagonal_term_kappa(10)

73*κ + 270

In [160]:
[(m, diagonal_term_kappa(m)) for m in range(2, 11+1)]

[(2, κ + 6),
 (3, 3*κ + 18),
 (4, 7*κ + 36),
 (5, 13*κ + 60),
 (6, 21*κ + 90),
 (7, 31*κ + 126),
 (8, 43*κ + 168),
 (9, 57*κ + 216),
 (10, 73*κ + 270),
 (11, 91*κ + 330)]

In [161]:
# Build a hypothesis
[(m, diagonal_term_kappa(m) - (m^2 - 3*m + 3) * κ) for m in range(2, 11+1)]

[(2, 6),
 (3, 18),
 (4, 36),
 (5, 60),
 (6, 90),
 (7, 126),
 (8, 168),
 (9, 216),
 (10, 270),
 (11, 330)]

In [162]:
# Build a hypothesis
[(m, (diagonal_term_kappa(m) - (m^2 - 3*m + 3) * κ)/m/(m-1)) for m in range(2, 11+1)]

[(2, 3),
 (3, 3),
 (4, 3),
 (5, 3),
 (6, 3),
 (7, 3),
 (8, 3),
 (9, 3),
 (10, 3),
 (11, 3)]

In [163]:
# Test hypothesis
[(m, (diagonal_term_kappa(m) - ((m^2 - 3*m + 3) * κ + 3 * m * (m-1)) )) for m in range(2, 11+1)]

[(2, 0),
 (3, 0),
 (4, 0),
 (5, 0),
 (6, 0),
 (7, 0),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 0)]

In [200]:
def diagonal_term_hyp(N):
    return σ^4 / N^3 * (N-1) * ((N^2 - 3*N + 3) * κ + 3 * N * (N - 1))

def diagonal_term(N):
    X = get_Xs(N)
    Xbar = get_Xbar(X)
    return expectation((X[1]-Xbar)^4).factor()

[(m, diagonal_term_hyp(m) - diagonal_term(m)) for m in range(2, 11+1)]

[(2, 0),
 (3, 0),
 (4, 0),
 (5, 0),
 (6, 0),
 (7, 0),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 0)]

In [202]:
def cross_term_hyp(N):
    return σ^4 / N^3 * ((2*N-3) * κ + N*(N^2 - 2*N + 3))

def cross_term(N):
    X = get_Xs(N)
    Xbar = get_Xbar(X)
    return expectation((X[1]-Xbar)^2*(X[2]-Xbar)^2).factor()

[(m, cross_term_hyp(m) - cross_term(m)) for m in range(2, 11+1)]

[(2, 0),
 (3, 0),
 (4, 0),
 (5, 0),
 (6, 0),
 (7, 0),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 0)]

In [166]:
def cross_term_hyp(N):
    "alternate hypothesis"
    return σ^4 * (1 - (N - κ)*(2*N - 3) / N^3)

def cross_term(N):
    X = get_variables(N, prefix='X_', ring=(n[1]/n[2]).parent())
    Xbar = sum([X[i] for i in range(1, N+1)]) / N
    return expectation((X[1]-Xbar)^2*(X[2]-Xbar)^2).factor()

[(m, cross_term_hyp(m) - cross_term(m)) for m in range(3, 11+1)]

[(3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0), (10, 0), (11, 0)]

## Checking some calculations

In [203]:
n = var('n')
# This is wrong, but I wanted to check how far off we were. Closer!
e4 = 1/n^4*(
    2*(n-1)^2*sample_moment4(1) + (n-2)^4*sample_moment4(n-2)
    -4*(n-1)*((n-1)^2+1)*sample_moment3(1)*sample_moment1(1) + 4*(n-1)*(n-2)^2*sample_moment3(1)*sample_moment1(n-2) - 4*(n-2)^4*sample_moment1(1)*sample_moment3(n-2)
    +((n-1)^4 + 4*(n-1)^2 + 1)*sample_moment2(1)^2 + 2*(n-2)^2*((n-1)^2 - 4*(n-1) + 1)*sample_moment2(1)*sample_moment2(n-2)
    -4*(n-2)*((n-1)^3 - 2*(n-1)^2 + 2*(n-1) - 1)*sample_moment2(1)*sample_moment1(1)*sample_moment1(n-2)
    +4*(n-2)^2*((n-1)^2 - (n-1) + 1)*sample_moment1(1)^2*sample_moment2(n-2)
)
e4 = e4.full_simplify()
show(e4)

(n^3 - 2*n^2 + (2*n - 3)*κ + 3*n)*σ^4/n^3

In [168]:
e4.coefficient(σ, 4).factor()

(n^3 - 2*n^2 + 2*n*κ + 3*n - 3*κ)/n^3

In [169]:
e4.coefficient(σ, 4).coefficient(κ, 0).factor()

(n^2 - 2*n + 3)/n^2

Check $\gamma_1\mu\sigma^3$ coefficient.

In [170]:
e5 = 8*(n-1)^2 + 4*(n-2)^2 - 4*(n-1)*((n-1)^2 + 1) + 4*(n-1)*(n-2)^2 - 4*(n-2)^2
e5.full_simplify()

0

In [171]:
# term = 𝛾_1*𝜇*𝜎^3
# term = 𝜇^2*𝜎^2
term = 𝜇^4
show(term)
e4a = [
    (part / n^4).full_simplify().coefficient(term, 1).full_simplify().factor() for
    part in [
        2*(n-1)^2*sample_moment4(1),
        (n-2)^4*sample_moment4(n-2),
        -4*(n-1)*((n-1)^2+1)*sample_moment3(1)*sample_moment1(1),
        4*(n-1)*(n-2)^2*sample_moment3(1)*sample_moment1(n-2),
        -4*(n-2)^4*sample_moment1(1)*sample_moment3(n-2),
        ((n-1)^4 + 4*(n-1)^2 + 1)*sample_moment2(1)^2,
        2*(n-2)^2*((n-1)^2 - 4*(n-1) + 1)*sample_moment2(1)*sample_moment2(n-2),
        -4*(n-2)*((n-1)^3 - 2*(n-1)^2 + 2*(n-1) - 1)*sample_moment2(1)*sample_moment1(1)*sample_moment1(n-2),
        4*(n-2)^2*((n-1)^2 - (n-1) + 1)*sample_moment1(1)^2*sample_moment2(n-2),
    ]
]
show(e4a)

μ^4

[2*(n - 1)^2/n^4,
 (n - 2)^4/n^4,
 -4*(n^2 - 2*n + 2)*(n - 1)/n^4,
 4*(n - 1)*(n - 2)^2/n^4,
 -4*(n - 2)^4/n^4,
 (n^4 - 4*n^3 + 10*n^2 - 12*n + 6)/n^4,
 2*(n^2 - 6*n + 6)*(n - 2)^2/n^4,
 -4*(n^2 - 3*n + 3)*(n - 2)^2/n^4,
 4*(n^2 - 3*n + 3)*(n - 2)^2/n^4]

Check $\mu^2\sigma^2$ term.

In [172]:
sum(e4a).full_simplify()

0

In [173]:
e7 = (
    2*(n-1)^2
    +(n-2)^4
    -4*(n-1)*((n-1)^2+1)
    +4*(n-1)*(n-2)^2
    -4*(n-2)^4
    +((n-1)^4+4*(n-1)^2+1)
    +2*(n-2)^2*((n-1)^2-4*(n-1)+1)
    -4*(n-2)*((n-1)^3-2*(n-1)^2+2*(n-1)-1)
    +4*(n-2)^2*((n-1)^2-(n-1)+1)
)
e7 = e7.full_simplify()
show(e7)

0

In [174]:
e6 = (
    12*(n-1)^2 #check
    + 6*(n-2)^3 #check
    - 12*(n-1)*((n-1)^2 + 1) #check
    + 12*(n-1)*(n-2)^2 #check
    - 12*(n-2)^3 #check
    + 2*((n-1)^4 + 4*(n-1)^2 + 1) #check
    + 2*(n-1)*(n-2)*((n-1)^2-4*(n-1)+1) #check
    - 4*(n-2)*((n-1)^3-2*(n-1)^2+2*(n-1)-1) #check
    + 4*(n-2)*((n-1)^2-(n-1)+1) #check
)
e6 = e6.full_simplify()
show(e6)

0

In [175]:
((n-1)^3-2*(n-1)^2+2*(n-1)-1).factor()

(n^2 - 3*n + 3)*(n - 2)

## Full calculation

In [205]:
N = 10
X = get_Xs(N)
Xbar = get_Xbar(X)
S2 = sum([(X[i] - Xbar)^2 for i in range(1, N+1)]) / (N-1)
# S2 = S2.full_simplify()
show(S2)

1/10*X_1^2 + (-1/45)*X_1*X_2 + 1/10*X_2^2 + (-1/45)*X_1*X_3 + (-1/45)*X_2*X_3 + 1/10*X_3^2 + (-1/45)*X_1*X_4 + (-1/45)*X_2*X_4 + (-1/45)*X_3*X_4 + 1/10*X_4^2 + (-1/45)*X_1*X_5 + (-1/45)*X_2*X_5 + (-1/45)*X_3*X_5 + (-1/45)*X_4*X_5 + 1/10*X_5^2 + (-1/45)*X_1*X_6 + (-1/45)*X_2*X_6 + (-1/45)*X_3*X_6 + (-1/45)*X_4*X_6 + (-1/45)*X_5*X_6 + 1/10*X_6^2 + (-1/45)*X_1*X_7 + (-1/45)*X_2*X_7 + (-1/45)*X_3*X_7 + (-1/45)*X_4*X_7 + (-1/45)*X_5*X_7 + (-1/45)*X_6*X_7 + 1/10*X_7^2 + (-1/45)*X_1*X_8 + (-1/45)*X_2*X_8 + (-1/45)*X_3*X_8 + (-1/45)*X_4*X_8 + (-1/45)*X_5*X_8 + (-1/45)*X_6*X_8 + (-1/45)*X_7*X_8 + 1/10*X_8^2 + (-1/45)*X_1*X_9 + (-1/45)*X_2*X_9 + (-1/45)*X_3*X_9 + (-1/45)*X_4*X_9 + (-1/45)*X_5*X_9 + (-1/45)*X_6*X_9 + (-1/45)*X_7*X_9 + (-1/45)*X_8*X_9 + 1/10*X_9^2 + (-1/45)*X_1*X_10 + (-1/45)*X_2*X_10 + (-1/45)*X_3*X_10 + (-1/45)*X_4*X_10 + (-1/45)*X_5*X_10 + (-1/45)*X_6*X_10 + (-1/45)*X_7*X_10 + (-1/45)*X_8*X_10 + (-1/45)*X_9*X_10 + 1/10*X_10^2

In [206]:
expectation(S2)

σ^2

In [207]:
varS2 = expectation(S2^2) - expectation(S2)^2
varS2 = varS2.full_simplify()
show(varS2)

1/90*(9*κ + 20)*σ^4

In [210]:
def varS2_hyp(N):
    return σ^4 / N * (κ + (2*N)/(N-1))

In [211]:
zero = varS2 - varS2_hyp(N)
zero = zero.full_simplify()
show(zero)

0

In [214]:
def S2(N):
    X = get_Xs(N)
    Xbar = get_Xbar(X)
    S2_ = sum([(X[i] - Xbar)^2 for i in range(1, N+1)]) / (N-1)
    return S2_
S2(2)

1/2*X_1^2 - X_1*X_2 + 1/2*X_2^2

In [215]:
def S2a(N):
    X = get_Xs(N)
    S2_ = sum([(X[i] - X[j])^2 for i in range(1, N+1) for j in range(1, N+1)]) / (2*N*(N-1))
    return S2_
S2a(2)

1/2*X_1^2 - X_1*X_2 + 1/2*X_2^2

In [216]:
# Test that these are equivalent
for N in range(2, 15):
    if S2(N) != S2a(N):
        show(N)
        show(S2(N))
        show(S2a(N))

In [217]:
def Var(expr):
    e = expectation(expr^2) - expectation(expr)^2
    e = e.full_simplify()
    return e

Var(S2(2))

1/2*(κ + 4)*σ^4

In [188]:
[(k, S2(k)) for k in range(2, 11)]

[(2, 1/2*X_1^2 - X_1*X_2 + 1/2*X_2^2),
 (3,
  1/3*X_1^2 + (-1/3)*X_1*X_2 + 1/3*X_2^2 + (-1/3)*X_1*X_3 + (-1/3)*X_2*X_3 + 1/3*X_3^2),
 (4,
  1/4*X_1^2 + (-1/6)*X_1*X_2 + 1/4*X_2^2 + (-1/6)*X_1*X_3 + (-1/6)*X_2*X_3 + 1/4*X_3^2 + (-1/6)*X_1*X_4 + (-1/6)*X_2*X_4 + (-1/6)*X_3*X_4 + 1/4*X_4^2),
 (5,
  1/5*X_1^2 + (-1/10)*X_1*X_2 + 1/5*X_2^2 + (-1/10)*X_1*X_3 + (-1/10)*X_2*X_3 + 1/5*X_3^2 + (-1/10)*X_1*X_4 + (-1/10)*X_2*X_4 + (-1/10)*X_3*X_4 + 1/5*X_4^2 + (-1/10)*X_1*X_5 + (-1/10)*X_2*X_5 + (-1/10)*X_3*X_5 + (-1/10)*X_4*X_5 + 1/5*X_5^2),
 (6,
  1/6*X_1^2 + (-1/15)*X_1*X_2 + 1/6*X_2^2 + (-1/15)*X_1*X_3 + (-1/15)*X_2*X_3 + 1/6*X_3^2 + (-1/15)*X_1*X_4 + (-1/15)*X_2*X_4 + (-1/15)*X_3*X_4 + 1/6*X_4^2 + (-1/15)*X_1*X_5 + (-1/15)*X_2*X_5 + (-1/15)*X_3*X_5 + (-1/15)*X_4*X_5 + 1/6*X_5^2 + (-1/15)*X_1*X_6 + (-1/15)*X_2*X_6 + (-1/15)*X_3*X_6 + (-1/15)*X_4*X_6 + (-1/15)*X_5*X_6 + 1/6*X_6^2),
 (7,
  1/7*X_1^2 + (-1/21)*X_1*X_2 + 1/7*X_2^2 + (-1/21)*X_1*X_3 + (-1/21)*X_2*X_3 + 1/7*X_3^2 + (-1/21)*X_1*X_4 +

In [189]:
show([(k, Var(S2(k))) for k in range(2, 11)])

[(2, 1/2*(κ + 4)*σ^4),
 (3, 1/3*(κ + 3)*σ^4),
 (4, 1/12*(3*κ + 8)*σ^4),
 (5, 1/10*(2*κ + 5)*σ^4),
 (6, 1/30*(5*κ + 12)*σ^4),
 (7, 1/21*(3*κ + 7)*σ^4),
 (8, 1/56*(7*κ + 16)*σ^4),
 (9, 1/36*(4*κ + 9)*σ^4),
 (10, 1/90*(9*κ + 20)*σ^4)]

In [218]:
def varS2_pub(size):
    return σ^4/size*(κ + (2*size)/(size-1))
varS2_pub(2)

1/2*(κ + 4)*σ^4

In [219]:
# Test that these are equivalent
for N in range(2, 15):
    v = Var(S2(N))
    v_pub = varS2_pub(N)
    if v != v_pub:
        show(N)
        show(v)
        show(v_pub)
        show((v-v_pub).full_simplify())

## Via characteristic functions and moment-generating functions

In [140]:
def mu(k):
    "define central moments"
    if k == 0:
        return 1
    elif k == 1:
        return 0
    elif k == 2:
        return σ^2
    elif k == 3:
        return γ*σ^3
    elif k == 4:
        return (κ + 3)*σ^4
    else:
        return var("μ_" + str(k))

def mu_prime(k):
    "define non-central moments in terms of central moments"
    return sum([
        binomial(k, j) * mu(j) * μ^(k-j)
        for j in range(0, k + 1)
    ])

t = var('t')
def M_X(t, up_to=4):
    "Moment-generating function up to order up_to"
    return (
        sum([
            mu_prime(k) * t^k / factorial(k)
            for k in range(0, up_to + 1)
        ])
        + t^(up_to + 1)*var('O_' + str(up_to + 1))
    ).full_simplify().collect(t)

def phi_X(t):
    "Characteristic function"
    return M_X(-i*t)

N = var('N', domain="integer")
def M_Xbar(N, t, up_to=4):
    "Moment-generating function for Xbar"
    return (M_X(t/N, up_to=up_to)^N).full_simplify().collect(t)

def phi_Xbar(N, t, up_to=4):
    "Characteristic function for Xbar"
    return (M_Xbar(N, -i*t, up_to=up_to))

def mu_prime_Xbar(N, k):
    "non-central moments for Xbar"
    return M_Xbar(N, t).derivative(t, k).substitute(t == 0).full_simplify()

In [142]:
show(M_Xbar(N, t))

(N*t^4*μ^4 + 4*N^2*t^3*μ^3 + 12*N^3*t^2*μ^2 + 24*O_5*t^5 + 24*N^4*t*μ + 24*N^5 + (N*t^4*κ + 3*N*t^4)*σ^4 + 4*(N*t^4*γ*μ + N^2*t^3*γ)*σ^3 + 6*(N*t^4*μ^2 + 2*N^2*t^3*μ + 2*N^3*t^2)*σ^2)^N/(24^N*N^(5*N))

In [143]:
show([(k, mu_prime_Xbar(N, k)) for k in range(1, 4 + 1)])

[(1, μ),
 (2, (N*μ^2 + σ^2)/N),
 (3, (N^2*μ^3 + 3*N*μ*σ^2 + γ*σ^3)/N^2),
 (4, (N^3*μ^4 + 6*N^2*μ^2*σ^2 + 4*N*γ*μ*σ^3 + (3*N + κ)*σ^4)/N^3)]

In [221]:
zeros = [(k, (mu_prime_Xbar(N, k) - sample_moment(k, N)).full_simplify()) for k in range(1, 4 + 1)]
show(zeros)

[(1, 0), (2, 0), (3, 0), (4, 0)]

## Checking $\text{Var}(S^2)$

In [234]:
# Right!
N = var('N')
varS2 = (1/(N-1))^2 * (N * diagonal_term_hyp(N) + N*(N-1)*cross_term_hyp(N)) - σ^4
varS2 = varS2.full_simplify()
show(varS2)
show(varS2_pub(N))

((N - 1)*κ + 2*N)*σ^4/(N^2 - N)

(κ + 2*N/(N - 1))*σ^4/N

In [235]:
show(diagonal_term_hyp(N))
show(cross_term_hyp(N))

(3*(N - 1)*N + (N^2 - 3*N + 3)*κ)*(N - 1)*σ^4/N^3

((N^2 - 2*N + 3)*N + (2*N - 3)*κ)*σ^4/N^3

In [197]:
var('a,b,k,n')
sum(binomial(n, k)*a^k*b^(n-k), k, 0, n)

(a + b)^n

In [225]:
# Trying another way:
N = var('N')
def term_hyp(i, j):
    if i == j:
        return diagonal_term_hyp(N)
    else:
        return cross_term_hyp(N)
def varS2_hyp(N):
    return (1/(N-1)^2*sum([term_hyp(i, j) for i in range(1, N + 1) for j in range(1, N + 1)]) - σ^4).full_simplify()


In [226]:
# Test that these are equivalent
for N in range(2, 15):
    v_hyp = varS2_hyp(N)
    v_pub = varS2_pub(N)
    if v_hyp != v_pub:
        show(N)
        show(v_hyp)
        show(v_pub)
        show((v_hyp-v_pub).full_simplify())

In [232]:
varS2_hyp(10).substitute(N==10).full_simplify()

1/90*(9*κ + 20)*σ^4